# Project 2 - Victoria rental data integration

This notebook integrates three complementary data layers for the three-year rental-price project: 2026 listing-level features from Kaggle, 2024-2025 suburb-level historical rents from Homes Victoria, and 2022-2026 affordability snapshots from Anglicare Victoria. The tables remain separate because their observational units differ.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'src' / 'integration.py').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.integration import load_victoria_data_layers, save_victoria_data_layers

FORECAST_HORIZON_YEARS = 3
layers = load_victoria_data_layers(PROJECT_ROOT)
listings = layers['kaggle_listings_vic_2026']
history = layers['homes_victoria_suburb_history']
affordability = layers['anglicare_affordability']
print(PROJECT_ROOT)
print(f'Forecast horizon: {FORECAST_HORIZON_YEARS} years')

C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate
Forecast horizon: 3 years


## 1. Validate coverage and observational units

In [2]:
inventory = pd.DataFrame({
    'dataset': ['Kaggle listings', 'Homes Victoria history', 'Anglicare affordability'],
    'observational_unit': ['individual listing', 'suburb x quarter x property type', 'Victoria annual snapshot'],
    'period': ['2026 snapshot', '2024 Q1 to 2025 Q3', '2022 to 2026'],
    'rows': [len(listings), len(history), len(affordability)],
})
inventory

,dataset,observational_unit,period,rows
0,Kaggle listings,individual listing,2026 snapshot,1118
1,Homes Victoria history,suburb x quarter x property type,2024 Q1 to 2025 Q3,7791
2,Anglicare affordability,Victoria annual snapshot,2022 to 2026,5


In [3]:
assert listings['state'].eq('VIC').all()
assert history['state'].eq('VIC').all()
assert listings['listing_id'].is_unique
assert history['period_end'].min() == pd.Timestamp('2024-03-31')
assert history['period_end'].max() == pd.Timestamp('2025-09-30')
print('All Victoria data-layer checks passed.')
print('Listing suburbs:', listings['suburb'].nunique())
print('Historical suburbs:', history['suburb'].nunique())
print('Historical periods:', history['period_end'].nunique())

All Victoria data-layer checks passed.
Listing suburbs: 335
Historical suburbs: 147
Historical periods: 7


## 2. Historical rental trend layer

In [4]:
history_summary = (
    history.groupby(['period_end', 'property_type'], as_index=False)
    .agg(
        suburbs_with_data=('suburb', 'nunique'),
        total_new_leases=('new_lease_count', 'sum'),
        median_of_suburb_medians=('median_weekly_rent_aud', 'median'),
    )
)
history_summary.tail(14)

,period_end,property_type,suburbs_with_data,total_new_leases,median_of_suburb_medians
35,2025-06-30,1 bedroom flat,147,66570.0,400.0
36,2025-06-30,2 bedroom flat,147,109260.0,500.0
37,2025-06-30,2 bedroom house,147,18057.0,518.0
38,2025-06-30,3 bedroom flat,147,28256.0,630.0
39,2025-06-30,3 bedroom house,147,88586.0,600.0
40,2025-06-30,4 bedroom house,147,71872.0,720.0
41,2025-06-30,All properties,147,412624.0,560.0
42,2025-09-30,1 bedroom flat,147,65051.0,400.0
43,2025-09-30,2 bedroom flat,147,107076.0,510.0
44,2025-09-30,2 bedroom house,147,18763.0,520.0


## 3. Save processed layers

The processed CSVs are ignored by Git because they are reproducible from the documented raw inputs.

In [5]:
processed_paths = save_victoria_data_layers(PROJECT_ROOT, layers)
for name, path in processed_paths.items():
    print(name, '->', path)

kaggle_listings_vic_2026 -> C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate\data\processed\kaggle_listings_vic_2026.csv
homes_victoria_suburb_history -> C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate\data\processed\homes_victoria_rents_2024_2025.csv
anglicare_affordability -> C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate\data\processed\anglicare_ras_victoria_2022_2026_summary.csv
